# Data preparation

## Running the Propp pipeline for every book in La comédie humaine

In [ ]:
def process_book(book_path, spacy_model, mentions_detection_model, coreference_resolution_model):

    ######################################################################################

    from propp_fr import load_text_file
    text_content = load_text_file(book_path)

    ######################################################################################

    from propp_fr import generate_tokens_df
    tokens_df = generate_tokens_df(text_content, spacy_model)

    ######################################################################################

    from propp_fr import load_tokenizer_and_embedding_model, get_embedding_tensor_from_tokens_df

    # Load the tokenizer and pre-trained embedding model
    tokenizer, embedding_model = load_tokenizer_and_embedding_model(
        mentions_detection_model["base_model_name"],
      )

    # Generate embeddings for all tokens
    tokens_embedding_tensor = get_embedding_tensor_from_tokens_df(
        text_content,
        tokens_df,
        tokenizer,
        embedding_model,
      )

    ######################################################################################

    from propp_fr import generate_entities_df

    entities_df = generate_entities_df(
        tokens_df,
        tokens_embedding_tensor,
        mentions_detection_model,
    )

    ######################################################################################

    from propp_fr import add_features_to_entities

    entities_df = add_features_to_entities(entities_df, tokens_df)

    ######################################################################################

    from propp_fr import perform_coreference

    entities_df = perform_coreference(
        entities_df,
        tokens_embedding_tensor,
        coreference_resolution_model,
        )

    ######################################################################################

    from propp_fr import extract_attributes
    tokens_df = extract_attributes(entities_df, tokens_df)

    ######################################################################################

    from propp_fr import generate_characters_dict

    characters_dict = generate_characters_dict(tokens_df, entities_df)

    ######################################################################################

    return tokens_df, entities_df, characters_dict

In [ ]:
import os
import gc
import datetime
import traceback
import torch
from propp_fr import load_models, save_tokens_df, save_entities_df, save_book_file

# Загружаем модели один раз для всех книг
spacy_model, mentions_detection_model, coreference_resolution_model = load_models()

for subdir, dirs, files in os.walk("la_comedie_humaine"):
    for file in files:
        if not file.endswith(".txt"):
            continue

        book_name = file[:-4]
        book_path = os.path.join(subdir, file)

        # Пропускаем уже обработанные файлы
        book_file_path = os.path.join(subdir, book_name + "_book_file.book")
        if os.path.exists(book_file_path):
            print(f"Skipping (already processed): {file}")
            continue

        print("####################################################################################")
        print(f"Processing file: {file}")
        print(f"Start time: {datetime.datetime.now()}")

        try:
            tokens, entities, characters = process_book(
                book_path, spacy_model, mentions_detection_model, coreference_resolution_model
            )
            save_tokens_df(tokens, book_name + "_tokens", subdir)
            save_entities_df(entities, book_name + "_entities", subdir)
            save_book_file(characters, book_name + "_book_file", subdir)
            print(f"End time: {datetime.datetime.now()}")
        except Exception as e:
            print(f"ERROR processing {file}: {e}")
            traceback.print_exc()
        finally:
            # Очищаем память после каждой книги (и после ошибок тоже)
            gc.collect()
            torch.cuda.empty_cache()

In [2]:
from propp_fr import load_tokens_df

tokens_df = load_tokens_df("balzac-55-FC-tenebreuse-affaire_tokens", "la_comedie_humaine/etudes_des_moeurs/scenes_de_la_vie_politique/balzac-55-FC-tenebreuse-affaire")

print(tokens_df.columns.tolist())

['paragraph_ID', 'sentence_ID', 'token_ID_within_sentence', 'token_ID_within_document', 'word', 'lemma', 'byte_onset', 'byte_offset', 'POS_tag', 'dependency_relation', 'syntactic_head_ID', 'morph', 'is_mention_head', 'char_att_agent', 'char_att_patient', 'char_att_mod', 'char_att_poss']


In [3]:
from propp_fr import load_book_file

characters_dict = load_book_file("balzac-55-FC-tenebreuse-affaire_book_file", "la_comedie_humaine/etudes_des_moeurs/scenes_de_la_vie_politique/balzac-55-FC-tenebreuse-affaire")

# print((characters_dict))



C:\Users\covre\PycharmProjects\balzac_comedie_humaine\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


propp_fr package loaded successfully.


In [25]:
print(characters_dict[0]['mentions']['proper'])


[{'n': 'michu', 'c': 131}, {'n': 'grévin', 'c': 18}, {'n': 'le bonhomme d’ hauteserre', 'c': 2}, {'n': 'd’ hauteserre', 'c': 2}, {'n': 'ci - devant marquis de chargebœuf', 'c': 1}, {'n': 'du chef de la maison de chargebœuf', 'c': 1}, {'n': 'monsieur de chargebœuf', 'c': 1}, {'n': 'le sénateur de l’ empire', 'c': 1}, {'n': 'le vieux marquis de chargebœuf', 'c': 1}, {'n': 'mon pauvre michu', 'c': 1}, {'n': 'le marquis de chargebœuf', 'c': 1}, {'n': 'un homme entièrement semblable à michu', 'c': 1}, {'n': 'monsieur pigoult', 'c': 1}, {'n': 'monsieur d’ hauteserre', 'c': 1}, {'n': 'le marquis de simeuse', 'c': 1}, {'n': 'au marquis de simeuse', 'c': 1}, {'n': 'cinq - cygne', 'c': 1}, {'n': 'l’ aîné des d’ hauteserre', 'c': 1}, {'n': 'des d’ hauteserre', 'c': 1}, {'n': 'adrien', 'c': 1}, {'n': 'l’ ancien piqueur de la maison de simeuse', 'c': 1}, {'n': 'beauvisage', 'c': 1}, {'n': 'l’ ancien régisseur de gondreville', 'c': 1}, {'n': 'l’ abbé goujet', 'c': 1}, {'n': 'de michu', 'c': 1}]
